In [2]:
import os
import subprocess

def frames_to_mp4_ffmpeg(
    frames_dir,
    out_path="zpose.mp4",
    fps=16,
    pattern="frame_%05d.png",  # e.g. frame_00001.png, frame_00002.png, ...
):
    """
    frames_dir: directory with numbered frames
    pattern: printf-style name pattern for ffmpeg
    """
    input_pattern = os.path.join(frames_dir, pattern)

    cmd = [
        "ffmpeg",
        "-y",
        "-framerate", str(fps),
        "-i", input_pattern,
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        out_path,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
import numpy as np
from typing import List

import cv2
from controlnet_aux.open_pose import util, draw_poses, PoseResult
from controlnet_aux.open_pose.body import Keypoint, BodyResult
import json
js = [json.loads(p)[0] for p in (
# # Caro
# """
# [{"people": [{"pose_keypoints_2d": [437.7201976776123, 289.4465560913086, 1.0, 463.83587354421616, 500.5945737361908, 1.0, 227.6834853887558, 510.59632194042206, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 699.9882616996765, 490.59282553195953, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 373.26448702812195, 221.65692937374115, 1.0, 499.95329761505127, 221.65692937374115, 1.0, 282.1374478340149, 254.99609005451202, 1.0, 577.74467253685, 256.1073954105377, 1.0], "face_keypoints_2d": [292.13919603824615, 219.43431866168976, 1.0, 292.13919603824615, 254.99609005451202, 1.0, 295.47311210632324, 290.5578614473343, 1.0, 301.0296388864517, 326.11963284015656, 1.0, 313.2539978027344, 359.45879352092743, 1.0, 334.3687995672226, 386.1301220655441, 1.0, 363.2627388238907, 408.35622918605804, 1.0, 396.60189950466156, 425.0258095264435, 1.0, 433.2749762535095, 430.58233630657196, 1.0, 468.8367476463318, 426.1371148824692, 1.0, 502.17590832710266, 408.35622918605804, 1.0, 529.958542227745, 386.1301220655441, 1.0, 551.0733439922333, 358.34748816490173, 1.0, 563.2977029085159, 326.11963284015656, 1.0, 567.7429243326187, 290.5578614473343, 1.0, 569.9655350446701, 254.99609005451202, 1.0, 568.8542296886444, 219.43431866168976, 1.0, 323.25574600696564, 204.9873490333557, 1.0, 342.14793705940247, 194.98560082912445, 1.0, 365.4853495359421, 193.87429547309875, 1.0, 388.8227620124817, 197.20821154117584, 1.0, 411.0488691329956, 201.65343296527863, 1.0, 465.5028315782547, 200.54212760925293, 1.0, 486.6176333427429, 194.98560082912445, 1.0, 508.84374046325684, 191.65168476104736, 1.0, 529.958542227745, 193.87429547309875, 1.0, 548.8507332801819, 203.87604367733002, 1.0, 437.7201976776123, 224.99084544181824, 1.0, 437.7201976776123, 247.21695256233215, 1.0, 437.7201976776123, 269.44305968284607, 1.0, 437.7201976776123, 292.7804721593857, 1.0, 407.7149530649185, 307.2274417877197, 1.0, 422.16192269325256, 310.5613578557968, 1.0, 437.7201976776123, 313.8952739238739, 1.0, 452.16716730594635, 310.5613578557968, 1.0, 466.6141369342804, 306.11613643169403, 1.0, 345.48185312747955, 224.99084544181824, 1.0, 364.3740441799164, 216.10040259361267, 1.0, 385.4888459444046, 217.21170794963837, 1.0, 402.15842628479004, 229.43606686592102, 1.0, 383.2662352323532, 232.7699829339981, 1.0, 363.2627388238907, 231.6586775779724, 1.0, 471.0593583583832, 229.43606686592102, 1.0, 487.7289386987686, 216.10040259361267, 1.0, 507.73243510723114, 214.98909723758698, 1.0, 525.5133208036423, 223.87954008579254, 1.0, 508.84374046325684, 231.6586775779724, 1.0, 489.95154941082, 231.6586775779724, 1.0, 373.26448702812195, 339.4552971124649, 1.0, 399.93581557273865, 337.2326864004135, 1.0, 426.60714411735535, 337.2326864004135, 1.0, 437.7201976776123, 338.3439917564392, 1.0, 449.94455659389496, 337.2326864004135, 1.0, 475.50457978248596, 336.1213810443878, 1.0, 501.06460297107697, 338.3439917564392, 1.0, 484.3950226306915, 355.01357209682465, 1.0, 463.2802208662033, 366.1266256570816, 1.0, 438.831503033638, 371.6831524372101, 1.0, 413.271479845047, 367.2379310131073, 1.0, 391.0453727245331, 356.12487745285034, 1.0, 379.9323191642761, 341.6779078245163, 1.0, 408.8262584209442, 343.9005185365677, 1.0, 437.7201976776123, 346.1231292486191, 1.0, 466.6141369342804, 342.789213180542, 1.0, 495.5080761909485, 340.5666024684906, 1.0, 467.7254422903061, 350.56835067272186, 1.0, 437.7201976776123, 355.01357209682465, 1.0, 407.7149530649185, 351.67965602874756, 1.0, 373.26448702812195, 221.65692937374115, 1.0, 499.95329761505127, 221.65692937374115, 1.0], "hand_left_keypoints_2d": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 604.4160010814667, 473.9232451915741, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 604.4160010814667, 489.48152017593384, 1.0, 574.4107564687729, 500.5945737361908, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 603.304695725441, 499.4832683801651, 1.0, 578.8559778928757, 505.0397951602936, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 597.7481689453125, 505.0397951602936, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "hand_right_keypoints_2d": null}], "canvas_height": 512, "canvas_width": 910}]
# """,
# # Claire
# """
# [{"people": [{"pose_keypoints_2d": [441.6459248010069, 230.43614020198584, 1.0, 463.85634075819206, 410.9462480721995, 1.0, 318.8828984194746, 387.9281806256622, 1.0, 305.9604745898396, 490.49991977338993, 1.0, 0.0, 0.0, 0.0, 608.8297830969095, 433.96431551873684, 1.0, 624.9828128839532, 491.30757126274204, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 396.4174413972844, 180.36174786215025, 1.0, 488.48971118343377, 177.13114190474153, 1.0, 345.53539756809676, 210.24485296818125, 1.0, 558.7553907570739, 203.78364105336368, 1.0], "face_keypoints_2d": [347.9583520361533, 183.59235381955904, 1.0, 348.7660035255054, 211.86015594688547, 1.0, 350.3813065042099, 240.93560956356419, 1.0, 355.227215440323, 269.2034116908908, 1.0, 364.91903331254923, 295.0482593501607, 1.0, 381.87971458894503, 317.662501052022, 1.0, 402.87865331210196, 337.04613679647446, 1.0, 426.30054650331545, 351.5838636048138, 1.0, 453.7606971412897, 355.6221210515747, 1.0, 481.2208477792641, 347.5456061580529, 1.0, 504.64274097047746, 331.3925763710091, 1.0, 524.0263767149299, 311.2012891372045, 1.0, 539.3717550126215, 286.97174445663893, 1.0, 548.2559213954955, 261.1268967973689, 1.0, 550.6788758635521, 232.8590946700424, 1.0, 551.4865273529042, 204.5912925427158, 1.0, 549.0635728848476, 177.13114190474153, 1.0, 357.65016990837955, 168.24697552186745, 1.0, 372.1878967167189, 158.55515764964122, 1.0, 388.3409265037626, 156.13220318158466, 1.0, 404.4939562908064, 157.74750616028905, 1.0, 419.8393345884979, 160.97811211769778, 1.0, 458.60660607740283, 160.1704606283456, 1.0, 475.56728735379875, 155.3245516922325, 1.0, 494.143271608899, 152.09394573482376, 1.0, 512.7192558639993, 153.7092487135281, 1.0, 529.6799371403952, 162.59341509640217, 1.0, 440.03062182230246, 181.16939935150248, 1.0, 440.03062182230246, 198.93773211725056, 1.0, 440.8382733116548, 216.7060648829986, 1.0, 440.8382733116548, 234.47439764874684, 1.0, 420.6469860778501, 244.16621552097308, 1.0, 431.14645543942845, 247.39682147838175, 1.0, 442.45357629035914, 250.62742743579042, 1.0, 454.5683486306418, 246.58916998902964, 1.0, 466.68312097092473, 243.35856403162074, 1.0, 376.2261541634798, 183.59235381955904, 1.0, 389.95622948246705, 173.9005359473328, 1.0, 406.10925926951074, 175.51583892603713, 1.0, 418.2240316097934, 185.20765679826337, 1.0, 404.4939562908064, 188.43826275567216, 1.0, 389.95622948246705, 188.43826275567216, 1.0, 465.0678179922203, 183.59235381955904, 1.0, 477.9902418218553, 172.2852329686284, 1.0, 495.75857458760345, 170.66992998992401, 1.0, 511.10395288529503, 178.74644488344592, 1.0, 496.5662260769557, 184.4000053089112, 1.0, 480.41319628991187, 185.20765679826337, 1.0, 390.76388097181916, 269.2034116908908, 1.0, 410.95516820562375, 265.972805733482, 1.0, 433.5694099074851, 266.78045722283423, 1.0, 443.26122777971125, 266.78045722283423, 1.0, 452.9530456519375, 265.1651542441299, 1.0, 480.41319628991187, 262.74219977607333, 1.0, 505.4503924598296, 264.35750275477767, 1.0, 492.52796863019466, 286.1640929672867, 1.0, 472.33668139638996, 305.5477287117392, 1.0, 445.6841822477678, 312.00894062655664, 1.0, 421.4546375672022, 307.1630316904435, 1.0, 403.68630480145407, 291.0100019033998, 1.0, 394.80213841858006, 271.62636615894735, 1.0, 418.2240316097934, 271.62636615894735, 1.0, 443.26122777971125, 272.4340176482996, 1.0, 473.1443328857422, 268.39576020153856, 1.0, 501.4121350130687, 266.78045722283423, 1.0, 477.9902418218553, 290.2023504140476, 1.0, 445.6841822477678, 299.0865167969217, 1.0, 416.6087286310891, 292.62530488210416, 1.0, 396.4174413972844, 180.36174786215025, 1.0, 488.48971118343377, 177.13114190474153, 1.0], "hand_left_keypoints_2d": null, "hand_right_keypoints_2d": null}], "canvas_height": 512, "canvas_width": 910}]
# """,
# Laurence
"""
[{"people": [{"pose_keypoints_2d": [235.18398028612137, 149.0063983798027, 1.0, 258.3633033633232, 242.27557933330536, 1.0, 150.19312900304794, 247.2425771355629, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 366.5334777235985, 237.30858153104782, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 203.174438893795, 123.61952072381973, 1.0, 263.33030116558075, 113.68552511930466, 1.0, 176.68378394842148, 154.52528482675552, 1.0, 314.655945122242, 134.65729361772537, 1.0], "face_keypoints_2d": [175.02811801433563, 132.44973903894424, 1.0, 177.23567259311676, 151.7658416032791, 1.0, 180.54700446128845, 169.97816687822342, 1.0, 185.514002263546, 187.63860350847244, 1.0, 193.24044328927994, 202.53959691524506, 1.0, 206.4857707619667, 216.3368130326271, 1.0, 220.83487552404404, 227.37458592653275, 1.0, 238.49531215429306, 233.99724966287613, 1.0, 256.70763742923737, 233.44536101818085, 1.0, 273.8161854147911, 226.27080863714218, 1.0, 288.7171788215637, 214.12925845384598, 1.0, 298.0992857813835, 200.8839309811592, 1.0, 305.82572680711746, 183.77538299560547, 1.0, 309.13705867528915, 166.66683501005173, 1.0, 309.68894731998444, 148.45450973510742, 1.0, 309.13705867528915, 131.34596174955368, 1.0, 306.929504096508, 113.13363647460938, 1.0, 179.99511581659317, 118.6525229215622, 1.0, 187.16966819763184, 112.5817478299141, 1.0, 196.55177515745163, 110.37419325113297, 1.0, 206.4857707619667, 109.82230460643768, 1.0, 216.41976636648178, 110.92608189582825, 1.0, 242.35853266716003, 105.95908409357071, 1.0, 252.2925282716751, 101.54397493600845, 1.0, 262.77841252088547, 98.23264306783676, 1.0, 273.8161854147911, 97.1288657784462, 1.0, 284.85395830869675, 100.99208629131317, 1.0, 231.3207597732544, 120.30818885564804, 1.0, 232.42453706264496, 130.24218446016312, 1.0, 234.0802029967308, 139.6242914199829, 1.0, 235.18398028612137, 149.558287024498, 1.0, 224.14620739221573, 160.59605991840363, 1.0, 230.7688711285591, 160.59605991840363, 1.0, 237.94342350959778, 161.1479485630989, 1.0, 245.66986453533173, 157.83661669492722, 1.0, 252.8444169163704, 155.6290621161461, 1.0, 192.13666599988937, 127.4827412366867, 1.0, 199.31121838092804, 121.4119661450386, 1.0, 208.69332534074783, 120.30818885564804, 1.0, 217.52354365587234, 124.7232980132103, 1.0, 209.24521398544312, 127.4827412366867, 1.0, 200.4149956703186, 128.58651852607727, 1.0, 249.5330850481987, 119.20441156625748, 1.0, 256.70763742923737, 112.02985918521881, 1.0, 266.64163303375244, 109.82230460643768, 1.0, 276.02373999357224, 112.5817478299141, 1.0, 267.745410323143, 116.99685698747635, 1.0, 258.9151920080185, 118.6525229215622, 1.0, 212.5565458536148, 182.6716057062149, 1.0, 222.49054145812988, 176.04894196987152, 1.0, 234.6320916414261, 173.2894987463951, 1.0, 240.7028667330742, 172.18572145700455, 1.0, 246.221753180027, 171.08194416761398, 1.0, 262.2265238761902, 169.42627823352814, 1.0, 277.6794059276581, 172.18572145700455, 1.0, 270.5048535466194, 184.87916028499603, 1.0, 259.4670806527138, 195.3650445342064, 1.0, 245.11797589063644, 200.33204233646393, 1.0, 231.87264841794968, 198.67637640237808, 1.0, 220.83487552404404, 192.0537126660347, 1.0, 215.31598907709122, 182.6716057062149, 1.0, 227.45753926038742, 178.80838519334793, 1.0, 241.25475537776947, 176.6008306145668, 1.0, 258.3633033633232, 173.2894987463951, 1.0, 274.91996270418167, 173.8413873910904, 1.0, 261.6746352314949, 185.9829375743866, 1.0, 244.01419860124588, 192.0537126660347, 1.0, 228.0094279050827, 190.39804673194885, 1.0, 203.174438893795, 123.61952072381973, 1.0, 263.33030116558075, 113.68552511930466, 1.0], "hand_left_keypoints_2d": null, "hand_right_keypoints_2d": null}], "canvas_height": 256, "canvas_width": 480}]
""",
"""
[{"people": [{"pose_keypoints_2d": [288.2636468410492, 151.7444441318512, 1.0, 280.6389303803444, 192.55910283327103, 1.0, 183.3116673231125, 158.92064785957336, 1.0, 81.94778966903687, 249.52021992206573, 1.0, 185.10571825504303, 238.75591433048248, 1.0, 377.9661934375763, 226.1975578069687, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 100.78532445430756, 224.40350687503815, 1.0, 99.88829898834229, 235.1678124666214, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 281.9844685792923, 105.99614536762238, 1.0, 331.3208692073822, 134.70096027851105, 1.0, 259.5588319301605, 87.15861058235168, 1.0, 361.8197350502014, 149.95039319992065, 1.0], "face_keypoints_2d": [263.1469337940216, 83.5705087184906, 1.0, 256.8677555322647, 95.23183977603912, 1.0, 251.48560273647308, 109.58424723148346, 1.0, 247.00047540664673, 125.73070561885834, 1.0, 245.2064244747162, 140.08311307430267, 1.0, 244.30939900875092, 158.92064785957336, 1.0, 247.00047540664673, 171.47900438308716, 1.0, 253.27965366840363, 186.72843730449677, 1.0, 264.94098472595215, 195.69869196414948, 1.0, 279.2933921813965, 199.28679382801056, 1.0, 294.5428251028061, 197.49274289608002, 1.0, 306.2041561603546, 191.21356463432312, 1.0, 322.3506145477295, 184.03736090660095, 1.0, 331.3208692073822, 175.06710624694824, 1.0, 343.879225730896, 164.302800655365, 1.0, 352.8494803905487, 153.53849506378174, 1.0, 360.92270958423615, 141.87716400623322, 1.0, 269.4261120557785, 86.26158511638641, 1.0, 278.3963667154312, 88.95266151428223, 1.0, 285.5725704431534, 94.33481431007385, 1.0, 292.74877417087555, 100.61399257183075, 1.0, 299.02795243263245, 106.89317083358765, 1.0, 321.4535890817642, 120.34855282306671, 1.0, 330.42384374141693, 123.03962922096252, 1.0, 340.2911238670349, 125.73070561885834, 1.0, 348.36435306072235, 129.31880748271942, 1.0, 355.5405567884445, 135.59798574447632, 1.0, 305.30713069438934, 122.14260375499725, 1.0, 299.02795243263245, 133.80393481254578, 1.0, 292.74877417087555, 145.4652658700943, 1.0, 286.46959590911865, 156.22957146167755, 1.0, 274.80826485157013, 152.64146959781647, 1.0, 278.3963667154312, 158.0236223936081, 1.0, 282.88149404525757, 162.50874972343445, 1.0, 289.16067230701447, 163.40577518939972, 1.0, 295.43985056877136, 164.302800655365, 1.0, 272.1171884536743, 99.71696710586548, 1.0, 282.88149404525757, 100.61399257183075, 1.0, 290.954723238945, 106.89317083358765, 1.0, 292.74877417087555, 116.76045095920563, 1.0, 283.77851951122284, 113.17234909534454, 1.0, 276.60231578350067, 107.79019629955292, 1.0, 317.86548721790314, 131.11285841464996, 1.0, 326.83574187755585, 128.42178201675415, 1.0, 336.7030220031738, 131.11285841464996, 1.0, 342.9822002649307, 140.08311307430267, 1.0, 334.011945605278, 140.98013854026794, 1.0, 325.0416909456253, 137.39203667640686, 1.0, 260.4558573961258, 154.435520529747, 1.0, 267.63206112384796, 160.7146987915039, 1.0, 274.80826485157013, 166.09685158729553, 1.0, 278.3963667154312, 168.78792798519135, 1.0, 282.88149404525757, 170.5819789171219, 1.0, 290.954723238945, 174.17008078098297, 1.0, 299.02795243263245, 177.75818264484406, 1.0, 290.954723238945, 179.5522335767746, 1.0, 281.9844685792923, 180.44925904273987, 1.0, 273.91123938560486, 176.86115717887878, 1.0, 266.7350356578827, 171.47900438308716, 1.0, 262.24990832805634, 163.40577518939972, 1.0, 262.24990832805634, 157.12659692764282, 1.0, 269.4261120557785, 165.19982612133026, 1.0, 276.60231578350067, 171.47900438308716, 1.0, 286.46959590911865, 175.06710624694824, 1.0, 296.33687603473663, 177.75818264484406, 1.0, 286.46959590911865, 175.9641317129135, 1.0, 276.60231578350067, 171.47900438308716, 1.0, 268.52908658981323, 165.19982612133026, 1.0, 281.9844685792923, 105.99614536762238, 1.0, 331.3208692073822, 134.70096027851105, 1.0], "hand_left_keypoints_2d": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 207.5313549041748, 182.2433099746704, 1.0, 203.94325304031372, 166.09685158729553, 1.0, 193.17894744873047, 147.25931680202484, 1.0, 203.04622757434845, 165.19982612133026, 1.0, 194.972998380661, 141.87716400623322, 1.0, 186.89976918697357, 140.08311307430267, 1.0, 181.51761639118195, 142.7741894721985, 1.0, 204.840278506279, 170.5819789171219, 1.0, 197.66407477855682, 154.435520529747, 1.0, 193.17894744873047, 156.22957146167755, 1.0, 198.5611002445221, 166.09685158729553, 1.0, 204.840278506279, 184.03736090660095, 1.0, 201.2521766424179, 172.37602984905243, 1.0, 202.14920210838318, 182.2433099746704, 1.0, 204.840278506279, 184.93438637256622, 1.0, 203.94325304031372, 214.53622674942017, 1.0, 203.94325304031372, 214.53622674942017, 1.0, 204.840278506279, 221.71243047714233, 1.0, 207.5313549041748, 221.71243047714233, 1.0], "hand_right_keypoints_2d": [185.10571825504303, 236.96186339855194, 1.0, 204.840278506279, 215.43325221538544, 1.0, 207.5313549041748, 189.41951370239258, 1.0, 216.50160956382751, 173.2730553150177, 1.0, 220.98673689365387, 162.50874972343445, 1.0, 173.4443871974945, 184.03736090660095, 1.0, 183.3116673231125, 168.78792798519135, 1.0, 201.2521766424179, 163.40577518939972, 1.0, 214.70755863189697, 161.61172425746918, 1.0, 173.4443871974945, 195.69869196414948, 1.0, 194.972998380661, 180.44925904273987, 1.0, 218.29566049575806, 179.5522335767746, 1.0, 228.16294062137604, 181.34628450870514, 1.0, 177.92951452732086, 209.15407395362854, 1.0, 200.35515117645264, 195.69869196414948, 1.0, 221.88376235961914, 194.8016664981842, 1.0, 232.6480679512024, 194.8016664981842, 1.0, 184.20869278907776, 222.6094559431076, 1.0, 203.94325304031372, 213.6392012834549, 1.0, 220.0897114276886, 210.0510994195938, 1.0, 230.85401701927185, 209.15407395362854, 1.0]}], "canvas_height": 256, "canvas_width": 480}]
""",
# """
# [{"people": [{"pose_keypoints_2d": [450.62808854691684, 223.159574797377, 1.0, 190.67465583514422, 477.5226111067459, 1.0, 16.44063462689519, 472.86394743807614, 1.0, 0.0, 0.0, 0.0, 661.19968637079, 504.5428603850305, 1.0, 364.90867704339325, 482.18127477541566, 1.0, 0.0, 0.0, 0.0, 663.0631518382579, 506.4063258524984, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 372.36253891326487, 174.70947264321148, 1.0, 450.62808854691684, 191.48066185042262, 1.0, 200.9237159062177, 230.6134366672486, 1.0, 458.08195041678846, 243.65769493952394, 1.0], "face_keypoints_2d": [221.42183604836464, 193.34412731789052, 1.0, 223.28530151583254, 232.4769021347165, 1.0, 225.14876698330045, 267.8827460166067, 1.0, 232.60262885317206, 307.0155208334327, 1.0, 243.7834216579795, 340.55789924785495, 1.0, 268.00847273506224, 374.1002776622772, 1.0, 288.5065928772092, 402.0522596742958, 1.0, 320.18550582416356, 428.14077621884644, 1.0, 353.72788423858583, 446.7754308935255, 1.0, 389.133728120476, 448.6388963609934, 1.0, 418.9491755999625, 431.86770715378225, 1.0, 437.5838302746415, 394.59839780442417, 1.0, 448.76462307944894, 346.14829565025866, 1.0, 461.80888135172427, 307.0155208334327, 1.0, 465.5358122866601, 264.1558150816709, 1.0, 465.5358122866601, 228.7499711997807, 1.0, 463.67234681919217, 198.93452372029424, 1.0, 325.77590222656727, 143.03055969625711, 1.0, 348.1374878361821, 133.7132323589176, 1.0, 370.49907344579697, 133.7132323589176, 1.0, 390.9971935879439, 139.3036287613213, 1.0, 411.49531373009086, 148.62095609866083, 1.0, 441.3107612095773, 165.39214530587196, 1.0, 446.90115761198103, 165.39214530587196, 1.0, 452.49155401438475, 167.25561077333987, 1.0, 458.08195041678846, 169.11907624080777, 1.0, 461.80888135172427, 174.70947264321148, 1.0, 433.8568993397057, 184.026799980551, 1.0, 443.1742266770452, 197.07105825282633, 1.0, 456.21848494932055, 208.25185105763376, 1.0, 465.5358122866601, 219.43264386244118, 1.0, 413.35877919755876, 243.65769493952394, 1.0, 426.4030374698341, 245.52116040699184, 1.0, 443.1742266770452, 249.24809134192765, 1.0, 456.21848494932055, 251.11155680939555, 1.0, 467.399277754128, 247.38462587445974, 1.0, 344.4105569012463, 174.70947264321148, 1.0, 359.31828064098954, 170.98254170827568, 1.0, 377.9529353156686, 172.84600717574358, 1.0, 389.133728120476, 182.1633345130831, 1.0, 374.2260043807328, 182.1633345130831, 1.0, 359.31828064098954, 180.2998690456152, 1.0, 445.03769214451313, 193.34412731789052, 1.0, 452.49155401438475, 191.48066185042262, 1.0, 458.08195041678846, 195.20759278535843, 1.0, 459.94541588425636, 202.66145465523005, 1.0, 456.21848494932055, 200.79798918776214, 1.0, 450.62808854691684, 198.93452372029424, 1.0, 361.18174610845745, 308.8789863009006, 1.0, 396.5875899903476, 275.3366078864783, 1.0, 437.5838302746415, 266.0192805491388, 1.0, 446.90115761198103, 267.8827460166067, 1.0, 452.49155401438475, 271.6096769515425, 1.0, 461.80888135172427, 297.69819349609315, 1.0, 445.03769214451313, 334.96750284545124, 1.0, 445.03769214451313, 357.3290884550661, 1.0, 437.5838302746415, 375.9637431297451, 1.0, 420.8126410674304, 385.28107046708465, 1.0, 389.133728120476, 370.3733467273414, 1.0, 368.63560797832906, 340.55789924785495, 1.0, 364.90867704339325, 308.8789863009006, 1.0, 402.17798639275134, 280.927004288882, 1.0, 441.3107612095773, 280.927004288882, 1.0, 450.62808854691684, 299.56165896356106, 1.0, 443.1742266770452, 333.10403737798333, 1.0, 437.5838302746415, 361.0560193900019, 1.0, 420.8126410674304, 368.5098812598735, 1.0, 383.5433317180723, 348.01176111772656, 1.0, 372.36253891326487, 174.70947264321148, 1.0, 450.62808854691684, 191.48066185042262, 1.0], "hand_left_keypoints_2d": [679.834341045469, 500.8159294500947, 1.0, 657.4727554358542, 487.77167117781937, 1.0, 635.1111698262393, 478.45434384047985, 1.0, 640.7015662286431, 463.5466201007366, 1.0, 655.6092899683863, 441.18503449112177, 1.0, 655.6092899683863, 452.3658272959292, 1.0, 653.7458245009184, 424.41384528391063, 1.0, 692.8785993177444, 403.9157251417637, 1.0, 711.5132539924234, 387.14453593455255, 1.0, 681.697806512937, 457.9562236983329, 1.0, 698.4689957201481, 431.86770715378225, 1.0, 722.6940467972308, 415.0965179465711, 1.0, 741.3287014719099, 407.6426560766995, 1.0, 704.0593921225518, 463.5466201007366, 1.0, 717.1036503948271, 446.7754308935255, 1.0, 737.6017705369741, 428.14077621884644, 1.0, 745.0556324068457, 415.0965179465711, 1.0, 732.0113741345704, 469.13701650314033, 1.0, 737.6017705369741, 456.092758230865, 1.0, 745.0556324068457, 441.18503449112177, 1.0, 746.9190978743136, 422.5503798164427, 1.0], "hand_right_keypoints_2d": [0.0, 0.0, 0.0, 670.5170137081295, 484.04474024288356, 1.0, 666.7900827731937, 471.00048197060823, 1.0, 653.7458245009184, 450.5023618284613, 1.0, 0.0, 0.0, 0.0, 687.2882029153407, 459.8196891658008, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 696.6055302526802, 461.6831546332687, 1.0, 704.0593921225518, 435.59463808871806, 1.0, 718.967115862295, 424.41384528391063, 1.0, 741.3287014719099, 416.959983414039, 1.0, 698.4689957201481, 465.4100855682045, 1.0, 711.5132539924234, 446.7754308935255, 1.0, 726.4209777321666, 435.59463808871806, 1.0, 745.0556324068457, 428.14077621884644, 1.0, 705.9228575900197, 472.86394743807614, 1.0, 717.1036503948271, 461.6831546332687, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}], "canvas_height": 512, "canvas_width": 910}]
# """,
)]
h, w = js[0]["canvas_height"], js[0]["canvas_width"]
ks = [np.asarray(j["people"][0]["pose_keypoints_2d"]).reshape(-1, 3) for j in js]

def bb(a):
    print(a)
    a = a[a[:, 2] > 0, :2]
    print(a)
    xymin = a.min(axis=0)
    xymax = a.max(axis=0)
    return xymin, xymax
sel = [0,1,2,5,14,15,16,17]
bbs = [bb(k[sel,:]) for k in ks]
print(bbs)

cts = [((bbs[i][1] + bbs[i][0])) * 0.5 for i in range(2)]

for i in range(2):
    ks[i][~np.isin(np.arange(18), sel), 2] = 0
    md = np.max(bbs[i][1] - bbs[i][0])
    sc = 1.0
    ks[i][sel,:2] = (cts[0] - cts[i] + (ks[i][sel,:2]) * sc - (w/2,h/2)) / (h, h) + 0.5

# h = h // 2 // 16 * 16
# w = w // 2 // 16 * 16
# h, w = 720, 720
print(h, w)
# ks[0][~np.isin(np.arange(18), [0,14,15,16,17]), 2] = 0
# ks[1][5:,2] = 0
for i, s in enumerate(np.linspace(0, 1, 49)):
    t = np.clip(s * 1, 0.0, 1.0)
    body_xyc = (ks[0] * (1-t) + ks[1] * t)
    if t != 0 and t != 1:
        body_xyc[:,2] *= ks[0][:,2] * ks[1][:,2]
    keypoints: List[Keypoint] = []
    for idx, (x, y, s) in enumerate(body_xyc):
        if s <= 0.0:
            keypoints.append(None)
        else:
            keypoints.append(Keypoint(x=float(x), y=float(y), score=float(s), id=idx))
    body_result = BodyResult(
        keypoints=keypoints,
        total_score=float(body_xyc[:, 2].sum()),
        total_parts=int((body_xyc[:, 2] > 0).sum()),
    )

    pose = PoseResult(
        body=body_result,
        left_hand=None,
        right_hand=None,
        face=None,
    )

    # This uses util.draw_bodypose / draw_handpose / draw_facepose internally
    canvas = draw_poses([pose], h, w, draw_body=True, draw_hand=False, draw_face=False)
    cv2.imwrite(f"/tmp/frame_{i:03d}.png", canvas)

frames_to_mp4_ffmpeg(
    "/tmp",
    out_path="zpose.mp4",
    pattern="frame_%03d.png",  # e.g. frame_00001.png, frame_00002.png, ...
)
import glob 
for f in glob.glob("/tmp/frame_*.png"):
    os.remove(f)

AttributeError: module 'pkgutil' has no attribute 'ImpImporter'